# Qwen2.5-3B Tool-Calling QLoRA

Trains only a LoRA adapter for validated `tool_calls` and safe no-tool responses. Attach the private dataset `homing-hub-data-lora`, select GPU T4 x2, enable Internet, and run every cell from top to bottom. The run intentionally uses one T4: Qwen2.5-3B QLoRA fits in 16 GB and this avoids unsafe multi-GPU model sharding.

In [ ]:
# Requires Settings > Internet = On. Restart the session after this cell if Kaggle asks.
%pip install -q -U 'transformers>=4.46,<5' 'peft>=0.13,<1' 'accelerate>=1,<2' 'bitsandbytes>=0.43' 'datasets>=3,<4' 'safetensors>=0.4'

In [ ]:
import json
import os
# Must be set before importing/initializing PyTorch. Kaggle T4 x2 otherwise
# makes Trainer use DataParallel, which is incompatible with this 4-bit QLoRA path.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
# Attached Kaggle private Dataset: cronus14/homing-hub-data-lora.
INPUT_DIR = Path('/kaggle/input/homing-hub-data-lora')
TRAIN_JSONL = INPUT_DIR / 'train.jsonl'
VALIDATION_JSONL = INPUT_DIR / 'validation.jsonl'
OUTPUT_DIR = Path('/kaggle/working/qwen25-tool-calls-lora')
MAX_LENGTH = 1024
SEED = 42

assert torch.cuda.is_available(), 'Choose GPU T4 x2 under Settings > Accelerator first.'
assert TRAIN_JSONL.exists() and VALIDATION_JSONL.exists(), f'Missing train/validation JSONL in {INPUT_DIR}'
# Optional Kaggle secret; do not paste a token in this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert torch.cuda.device_count() == 1, 'Restart the session so CUDA_VISIBLE_DEVICES=0 takes effect.'
print({'gpu_count': torch.cuda.device_count(), 'gpu_0': torch.cuda.get_device_name(0), 'input_files': sorted(path.name for path in INPUT_DIR.iterdir())})

In [ ]:
def load_records(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

train_records, validation_records = load_records(TRAIN_JSONL), load_records(VALIDATION_JSONL)
assert len(train_records) >= 600 and len(validation_records) >= 60, 'Dataset is unexpectedly small; check the attached files.'
for record in train_records + validation_records:
    assert [message['role'] for message in record['messages']] == ['user', 'assistant']
    if record['tool_call_expected']:
        assert record['messages'][1]['content'].startswith('```tool_calls\n')
    else:
        assert '```tool_calls' not in record['messages'][1]['content']
train_dataset = Dataset.from_list(train_records).shuffle(seed=SEED)
validation_dataset = Dataset.from_list(validation_records)
print(f'{len(train_dataset)} train / {len(validation_dataset)} validation examples; no benchmark holdout should be present.')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=False)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def tokenize(record):
    user, assistant = record['messages']
    prompt = tokenizer.apply_chat_template([user], tokenize=False, add_generation_prompt=True)
    full_text = prompt + assistant['content'] + tokenizer.eos_token
    full = tokenizer(full_text, max_length=MAX_LENGTH, truncation=True, add_special_tokens=False)
    prompt_len = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
    labels = full['input_ids'].copy()
    labels[:prompt_len] = [-100] * min(prompt_len, len(labels))
    full['labels'] = labels
    return full

tokenized_train = train_dataset.map(tokenize, remove_columns=train_dataset.column_names)
tokenized_validation = validation_dataset.map(tokenize, remove_columns=validation_dataset.column_names)
assert all(any(label != -100 for label in row['labels']) for row in tokenized_train), 'Assistant targets were masked.'

In [ ]:
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
# Train on GPU 0 only. `device_map='auto'` can shard across T4s and is not a safe Trainer configuration.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN, quantization_config=quantization, device_map={'': 0}, torch_dtype=torch.float16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(output_dir=str(OUTPUT_DIR), num_train_epochs=3, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05, logging_steps=5, logging_first_step=True, save_strategy='epoch', eval_strategy='epoch', save_total_limit=2, fp16=True, bf16=False, gradient_checkpointing=True, optim='paged_adamw_8bit', report_to='none', seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=tokenized_train, eval_dataset=tokenized_validation, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100))
train_result = trainer.train()
trainer.save_state()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
(OUTPUT_DIR / 'training_metadata.json').write_text(json.dumps({'base_model': MODEL_ID, 'max_length': MAX_LENGTH, 'train_examples': len(train_dataset), 'validation_examples': len(validation_dataset), 'epochs': 3, 'train_metrics': train_result.metrics, 'eval_metrics': trainer.evaluate()}, ensure_ascii=False, indent=2), encoding='utf-8')
print(OUTPUT_DIR)

In [ ]:
!cd /kaggle/working && zip -r qwen25-tool-calls-lora.zip qwen25-tool-calls-lora
print('Download /kaggle/working/qwen25-tool-calls-lora.zip as a Kaggle output artifact.')